# FutureWork AI — Phase 2: Explainable Career Recommendations
**IBM LinuxONE Datathon 2026 · Tech for Good**

This notebook reads the **four CSV exports of Phase 1** and calculates career-adjacency recommendations using O*NET occupational **skill importance ratings**. Similarity measures similarity between *job profiles*, **not a person's likelihood of getting hired or their actual proficiency**.

**Data attribution:** O*NET 31.0 Database, U.S. Department of Labor/Employment and Training Administration, CC BY 4.0; transformed in Phase 1. https://www.onetcenter.org/database.html

**Geographic caveat:** O*NET is US-based; results are not Australian career or qualification advice. We do not model qualifications, licences, wages or vacancies in this prototype.

## 0. Required files
Run Phase 1 successfully first. Put this notebook in the **same Jupyter working directory** as Phase 1. The folder `processed/` must contain:
- `occupation_skill_matrix.csv`
- `occupation_lookup.csv`
- `skill_dictionary.csv`
- `skills_long_clean.csv`

This notebook uses pandas, numpy, matplotlib and scikit-learn (if the last package is absent, install it in your Jupyter environment).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
PROCESSED = ROOT / 'processed'
required = ['occupation_skill_matrix.csv', 'occupation_lookup.csv', 'skill_dictionary.csv', 'skills_long_clean.csv']
missing = [name for name in required if not (PROCESSED / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing Phase 1 outputs: {missing}. Run Phase 1 first; make sure notebooks are in the same folder. Checked {PROCESSED}')
print('Using Phase 1 exports in:', PROCESSED)

## 1. Load and validate Phase 1 outputs

In [ ]:
raw = pd.read_csv(PROCESSED / 'occupation_skill_matrix.csv', dtype={'occupation_code': str})
lookup = pd.read_csv(PROCESSED / 'occupation_lookup.csv', dtype={'occupation_code': str})
dictionary = pd.read_csv(PROCESSED / 'skill_dictionary.csv')
long_table = pd.read_csv(PROCESSED / 'skills_long_clean.csv', dtype={'O*NET-SOC Code': str})
assert 'occupation_code' in raw and 'occupation_title' in lookup
assert {'skill_id', 'skill_name'}.issubset(dictionary.columns)
assert not raw['occupation_code'].duplicated().any()
raw = raw.set_index('occupation_code')
raw = raw.apply(pd.to_numeric, errors='coerce')
lookup = lookup.drop_duplicates('occupation_code').set_index('occupation_code')
features = list(raw.columns.intersection(dictionary['skill_id'].astype(str)))
assert len(features) >= 2, 'Not enough matching skills between matrix and dictionary.'
X = raw.loc[:, features].copy()
common = X.index.intersection(lookup.index)
X = X.loc[common]
lookup = lookup.loc[common]
assert X.shape[0] >= 3
assert X.notna().any().any()
assert (X.stack().between(1, 5)).all(), 'Skill importance outside expected 1–5 range.'
names = dictionary.drop_duplicates('skill_id').set_index('skill_id')['skill_name'].to_dict()
print('Occupations:', len(X), '| Skill dimensions:', X.shape[1])
print('Missing ratings:', f'{X.isna().mean().mean():.1%}')
display(lookup.head())

## 2. Prepare features for similarity
Missing occupation ratings are filled with each skill's **median across occupations** solely for numerical ranking, not shown as actual ratings in explanations. All skill columns are on a shared importance scale (1–5). We subtract a baseline of 1 before computing cosine similarity, so the common minimum matters less. This is a heuristic, not a validated transition probability.

Very sparse occupations are excluded (at least 60% of skill dimensions must be observed), and we display the data coverage for chosen results.

In [ ]:
coverage = X.notna().mean(axis=1)
MIN_COVERAGE = 0.60
eligible = coverage[coverage >= MIN_COVERAGE].index
if len(eligible) < 3:
    raise ValueError('Too few occupations meet data coverage threshold. Review Phase 1 datasets.')
X_eligible = X.loc[eligible]
medians = X_eligible.median(axis=0).fillna(1.0)
filled = X_eligible.fillna(medians)
vector = np.maximum(filled.to_numpy(dtype=float) - 1.0, 0.0)
assert np.isfinite(vector).all()
SIM = pd.DataFrame(cosine_similarity(vector), index=eligible, columns=eligible)
print(f'Eligible occupations with ≥{MIN_COVERAGE:.0%} coverage:', len(eligible))
print('Similarity range:', round(float(SIM.min().min()), 3), 'to', round(float(SIM.max().max()), 3))

## 3. Recommendation and skill-explanation functions

In [ ]:
def find_occupations(query, limit=15):
    """Search titles; returns occupation codes and titles."""
    mask = lookup['occupation_title'].str.contains(str(query), case=False, na=False, regex=False)
    result = lookup.loc[mask, ['occupation_title']].reset_index()
    return result.head(limit)


def explain_pair(source_code, target_code, count=5):
    """Compare observed importance ratings only; not human skills or credentials."""
    a = X.loc[source_code]
    z = X.loc[target_code]
    valid = a.notna() & z.notna()
    tab = pd.DataFrame({'skill_id': X.columns[valid], 'source_importance': a[valid].values,
                        'target_importance': z[valid].values})
    if tab.empty:
        return pd.DataFrame(), pd.DataFrame()
    tab['skill'] = tab['skill_id'].map(names).fillna(tab['skill_id'])
    tab['gap'] = tab['target_importance'] - tab['source_importance']
    # Common highly important skills: evidence of potential occupational transfer.
    common = tab.assign(shared=tab[['source_importance','target_importance']].min(axis=1))
    common = common.sort_values(['shared','target_importance'], ascending=False).head(count)
    # Higher requirements of target occupation relative to current occupation.
    gaps = tab.loc[tab['gap'] > 0].sort_values(['gap','target_importance'], ascending=False).head(count)
    return common[['skill','source_importance','target_importance']], gaps[['skill','source_importance','target_importance','gap']]


def recommend(source_code, top_n=5):
    source_code = str(source_code)
    if source_code not in SIM.index:
        raise ValueError('Occupation code not eligible or not found. Use find_occupations() and check coverage.')
    scores = SIM.loc[source_code].drop(source_code).sort_values(ascending=False).head(top_n)
    result = pd.DataFrame({'occupation_code': scores.index,
                           'occupation_title': lookup.loc[scores.index, 'occupation_title'].values,
                           'similarity': scores.values,
                           'data_coverage': coverage.loc[scores.index].values})
    return result.reset_index(drop=True)

print('Functions ready. Search example:')
display(find_occupations('sales'))

## 4. Select an occupation and generate top-five recommendations
Choose a code from your search results below. If this exact example is missing in your release, use any eligible occupation code from `find_occupations('sales')`.

In [ ]:
matches = find_occupations('sales')
if matches.empty:
    source_code = eligible[0]
else:
    available_matches = matches[matches['occupation_code'].isin(eligible)]
    source_code = available_matches.iloc[0]['occupation_code'] if not available_matches.empty else eligible[0]
print('Selected occupation:', lookup.loc[source_code, 'occupation_title'], '| Code:', source_code)
recommendations = recommend(source_code, 5)
display(recommendations.style.format({'similarity':'{:.3f}', 'data_coverage':'{:.0%}'}))

## 5. Explain a recommended career and show skill gaps

In [ ]:
target_code = recommendations.iloc[0]['occupation_code']
transferable, gaps = explain_pair(source_code, target_code, count=7)
print('From:', lookup.loc[source_code, 'occupation_title'])
print('To:', lookup.loc[target_code, 'occupation_title'])
print('\nShared high-importance skills (possible transfer points):')
display(transferable)
print('Skills more important in the target occupation (learning priorities to investigate):')
display(gaps)
print('Note: These differences compare job requirements, not individual proficiency.')

## 6. Visualise recommendations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_data = recommendations.iloc[::-1]
ax.barh(plot_data['occupation_title'], plot_data['similarity'])
ax.set_xlim(0, 1.0)
ax.set_xlabel('Cosine similarity of occupational importance profiles (not employability %)')
ax.set_title('FutureWork AI — career-adjacency matches')
plt.tight_layout()
plt.show()

## 7. Sanity checks and save results
We verify the selected occupation isn't recommended to itself, recommendation scores are bounded, and explanation differences are calculated from observed data. Real-world relevance still requires human evaluation.

In [ ]:
assert len(recommendations) == 5
assert source_code not in set(recommendations['occupation_code'])
assert recommendations['similarity'].between(0, 1.000001).all()
assert recommendations['similarity'].is_monotonic_decreasing
assert (gaps['gap'] > 0).all() if not gaps.empty else True
OUTPUT = ROOT / 'results'
OUTPUT.mkdir(exist_ok=True)
recommendations.to_csv(OUTPUT / 'sample_recommendations.csv', index=False)
transferable.to_csv(OUTPUT / 'sample_transferable_skills.csv', index=False)
gaps.to_csv(OUTPUT / 'sample_skill_gaps.csv', index=False)
print('Phase 2 finished. Results saved to:', OUTPUT)
print('No claims about hiring probabilities, salary or Australian certifications.')

## Next phase
Add user-entered **self-rated skill proficiency** separately from occupational importance; never automatically equate the two. Add interactive inputs, ranking evaluation, licensing filters, and optionally Australian occupation mappings. Keep **O*NET source attribution** in public demos and submissions.